In [5]:

import requests
import pandas as pd
import time
import random

target_domain = "taylorstitch.com"
base_url = f"https://{target_domain}/products.json"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

new_data_pool = []

print(f"Setup Complete")

Setup Complete


In [10]:
page = 1
print(f"Launching master capture pipeline against: {target_domain}...\n")

while True:
    url = f"{base_url}?page={page}&limit=100"
    res = requests.get(url, headers=headers)

    if res.status_code != 200:
        print(f"Connection blocked {page}.")
        break

    products = res.json().get("products", [])

    if not products:
        print("\nReached the end ")
        break

    print(f"Page {page}... {len(products)} products.")

    for item in products:
        new_data_pool.append({
            "Brand/Vendor": item.get("vendor", "Unknown"),
            "Product Title": item["title"],
            "Product Type": item.get("product_type", "General"),
            "Raw Price": item["variants"][0]["price"]
        })

    time.sleep(random.uniform(0.4, 0.8))
    page += 1

print(f"Complete! Stored  {len(new_data_pool)} rows data")

Launching master capture pipeline against: taylorstitch.com...

Page 1... 100 products.
Page 2... 100 products.
Page 3... 100 products.
Page 4... 100 products.
Page 5... 100 products.
Page 6... 100 products.
Page 7... 100 products.
Page 8... 100 products.
Page 9... 100 products.
Page 10... 100 products.
Page 11... 100 products.
Page 12... 100 products.
Page 13... 100 products.
Page 14... 100 products.
Page 15... 100 products.
Page 16... 100 products.
Page 17... 100 products.
Page 18... 100 products.
Page 19... 100 products.
Page 20... 100 products.
Page 21... 100 products.
Page 22... 100 products.
Page 23... 100 products.
Page 24... 100 products.
Page 25... 100 products.
Page 26... 100 products.
Page 27... 100 products.
Page 28... 100 products.
Page 29... 100 products.
Page 30... 100 products.
Page 31... 100 products.
Page 32... 100 products.
Page 33... 100 products.
Page 34... 100 products.
Page 35... 100 products.
Page 36... 100 products.
Page 37... 100 products.
Page 38... 6 product

In [12]:
df_new_raw = pd.DataFrame(new_data_pool)

print(f"NEW TARGET PROFILE: {df_new_raw.shape[0]} rows captured by {df_new_raw.shape[1]} columns\n")
df_new_raw

NEW TARGET PROFILE: 4806 rows captured by 4 columns



,Brand/Vendor,Product Title,Product Type,Raw Price
0,Taylor Stitch,The Davis Shirt in Indigo Raindrop Sashiko,Wovens,138.00
1,Taylor Stitch,The Tour Belt in Washed Saddle,Accessories,108.00
2,Taylor Stitch,The Bandana in Faded Black Stamp,Accessories,45.00
3,Taylor Stitch,The 6-Panel Trucker Cap in Emerald Mesh,Accessories,68.00
4,Taylor Stitch,The Mercer Cap in Bud Green Twill,Accessories,68.00
...,...,...,...,...
4801,Taylor Stitch,The Jack in Sun Bleached Denim,Wovens,98.00
4802,Taylor Stitch,The Jack in Indigo Star,Wovens,98.00
4803,Taylor Stitch,The California in Indigo Pyramid,Wovens,75.00
4804,Taylor Stitch,The Jack in Black Everyday Oxford,Wovens,49.00


In [13]:
df_clean_step = df_new_raw.copy()

df_clean_step["Price (USD)"] = df_clean_step["Raw Price"].astype(float)

df_final_clean = df_clean_step.drop_duplicates(subset=["Product Title"], keep="first").copy()

def generate_brand_sku(row):
    clean_title = row["Product Title"].upper().replace("THE ", "").strip()
    return f"INF-TYS-{clean_title[:3]}-2026"

df_final_clean["Internal SKU"] = df_final_clean.apply(generate_brand_sku, axis=1)

final_taylor_catalog = df_final_clean[["Internal SKU", "Brand/Vendor", "Product Type", "Product Title", "Price (USD)"]]

final_taylor_catalog.to_csv("taylor_stitch_master_catalog.csv", index=False)

print(f"Raw Data: {len(df_new_raw)}")
print(f"Final Clean Products:  {len(final_taylor_catalog)}")
print(f"Duplicate Rows: {len(df_new_raw) - len(final_taylor_catalog)}")
print(f"File saved to: 'taylor_stitch_master_catalog.csv'")
final_taylor_catalog.head(5)

Raw Data: 4806
Final Clean Products:  3659
Duplicate Rows: 1147
File saved to: 'taylor_stitch_master_catalog.csv'


,Internal SKU,Brand/Vendor,Product Type,Product Title,Price (USD)
0,INF-TYS-DAV-2026,Taylor Stitch,Wovens,The Davis Shirt in Indigo Raindrop Sashiko,138.0
1,INF-TYS-TOU-2026,Taylor Stitch,Accessories,The Tour Belt in Washed Saddle,108.0
2,INF-TYS-BAN-2026,Taylor Stitch,Accessories,The Bandana in Faded Black Stamp,45.0
3,INF-TYS-6-P-2026,Taylor Stitch,Accessories,The 6-Panel Trucker Cap in Emerald Mesh,68.0
4,INF-TYS-MER-2026,Taylor Stitch,Accessories,The Mercer Cap in Bud Green Twill,68.0
